# SafeShare — train and measure on a free Colab GPU

Run the cells in order. Nothing here touches your laptop's copy of the project: this
session clones the repo, trains, and sends the results to your computer.

**The order matters.** The baseline is measured **before** any training — that is the
"before" column. If the baseline is already good, that is a result to report, not a problem.

**A Colab runtime is a fresh machine every session.** If it disconnects, `/content` is wiped —
code, the 6 GB model, the data, the adapter. Two rules follow: never leave a gap between the
long steps, and download the adapter the moment training ends. Cell 5 does both.

| step | cell | time on a T4 | can be skipped? |
|---|---|---|---|
| check the GPU, install, clone | 0–2 | 5 min | no |
| the base model + the training data | 3 | 5 min | no |
| Drive mount (optional) | 3a | — | yes |
| **baseline measurement (the "before")** | 4 | 5–10 min | no |
| **fine-tune, measure, download** | 5 | ~30 min | yes |
| align with DPO | 6 | ~25 min | yes |
| measure after alignment | 7 | 5–10 min | yes |
| the comparison table | 8 | seconds | no |
| final save | 9 | seconds | **no — do this one** |

Set the GPU first: **Runtime → Change runtime type → T4 GPU**.


In [ ]:
# 0 - is there really a GPU here?
!nvidia-smi

# You want 'Tesla T4' and about 15 GB. If it says no GPU: Runtime -> Change runtime type -> T4 GPU.


In [ ]:
# 1 - get the code
#
# The repository is private, so Colab needs permission to read it. Two ways:
#
#   (a) Recommended - add a GitHub token as a Colab secret named GITHUB_TOKEN:
#       sidebar key icon -> Add new secret -> Name: GITHUB_TOKEN
#       GitHub -> Settings -> Developer settings -> Personal access tokens -> Fine-grained
#       -> Repository access: only safeshare2 -> Permissions: Contents = Read-only
#   (b) Or make the repository public once. Verified: its history holds no secrets
#       (.env was never committed, and the only 'sk-' ever added was a placeholder).
#       Keeping it private is the safer choice for graded coursework.
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    repo = f'https://{token}@github.com/HajarOga/safeshare2.git'
    print('using the GITHUB_TOKEN from Colab secrets')
except Exception:
    repo = 'https://github.com/HajarOga/safeshare2.git'
    print('no GITHUB_TOKEN secret found - this clone only works if the repository is public')

!git clone -q $repo /content/safeshare2 || echo 'CLONE FAILED - see the two options above'
%cd /content/safeshare2

# do not leave the token inside the cloned folder's git config
!git remote set-url origin https://github.com/HajarOga/safeshare2.git
!ls


In [ ]:
# 2 - install what the app and the training need
#
# Colab ships its own versions of torch, pandas and friends, and this install moves some of
# them. Two consequences:
#
#   1. 'dependency resolver' warnings naming google-colab/cudf/dask/pandas - WARNINGS about
#      Colab's own packages. Harmless. What matters is the 'done' at the end.
#   2. Colab pairs torch with a matching torchvision. Installing our requirements can leave
#      the two out of step, and an out-of-step torchvision makes transformers fail to import
#      ANY model with 'operator torchvision::nms does not exist'. Nothing in this project
#      touches images, so removing torchvision is safe - and it removes that whole failure mode.
!pip install -q -r requirements.txt -r training/requirements-train.txt
!pip uninstall -y -q torchvision
print('done')


In [ ]:
# 3 - the base model (once per session), and the training data
import os

# local model, and let it fetch the weights it needs
os.environ['SAFESHARE2_PROVIDER'] = 'local'
os.environ['SAFESHARE2_ALLOW_DOWNLOAD'] = '1'

# ~6 GB, once. The app will not start without it - there is no rules-only fallback.
!python -m scripts.setup_model --yes

# the data: our own documents, then the prompt/completion and chosen/rejected pairs
!python -m training.make_training_documents
!python -m training.prepare_data --skip-public
!python -m training.make_preference_pairs


In [ ]:
# 3a - (optional) mount Google Drive, so a disconnect cannot cost you the run
#
# If this fails with 'credential propagation was unsuccessful': that is a known Colab
# authentication hiccup, not your code. Try once more with force_remount=True. If it still
# fails, skip it - NOTHING else in this notebook needs Drive. Use cell 10b at the end to
# download the results to your own computer instead.
MOUNTED = False
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    import os
    MOUNTED = os.path.isdir('/content/drive/MyDrive')
except Exception as exc:
    print('Drive mount failed:', exc)

if MOUNTED:
    !mkdir -p /content/drive/MyDrive/SafeShare/predictions
    print('Drive is ready - results will be saved there as they finish')
else:
    print('Continuing without Drive. Results stay in this Colab session:')
    print('  run cell 10b at the end to download them to your computer')


In [ ]:
# 4 - THE BASELINE: the prompt alone, no training at all
# This is the 'before' column. It needs a working GPU and about 5-10 minutes for 12 documents.
!python -m evaluation.evaluate --live --out predictions/base.jsonl
!python -m evaluation.evaluate --score predictions/base.jsonl --json predictions/base.metrics.json

import os
if os.path.isdir('/content/drive/MyDrive'):
    !mkdir -p /content/drive/MyDrive/SafeShare/predictions
    !cp predictions/base.jsonl predictions/base.metrics.json /content/drive/MyDrive/SafeShare/predictions/
    print('baseline saved to Drive')
else:
    print('Drive is not mounted - the baseline is in /content/safeshare2/predictions/.')
    print('Do not let this tab idle: run cell 10b at the end to download it.')

# If every document reports an error instead of results, check the model loaded at all:
#   !python -c "from transformers import AutoModelForCausalLM; print('transformers can load models')"


## Optional from here — skip the training and the table still works

If the baseline was already strong, you can stop after cell 4 and report that. The brief
still wants the adaptation step run, so continue when you have time.


In [ ]:
# 5 - fine-tune, measure, and download - ONE cell on purpose
#
# A Colab runtime is a fresh machine every time. A disconnect wipes the code, the 6 GB
# model, the data and the adapter. So the two long steps run without a gap (a running cell
# keeps the session alive), and the adapter comes down to your computer the moment it exists.
import os
os.environ['SAFESHARE2_PROVIDER'] = 'local'

# 1) train (~20 min, 108 pairs, 3 epochs)
!python -m training.train_lora --max-len 1536

# 2) measure the fine-tuned model on the test set it has never seen
os.environ['SAFESHARE2_ADAPTER'] = 'training/lora_adapter'
!python -m evaluation.evaluate --live --out predictions/lora.jsonl
!python -m evaluation.evaluate --score predictions/lora.jsonl

# 3) get it out before anything else can happen
import shutil
print('adapter present:', os.path.isdir('training/lora_adapter'))
shutil.make_archive('/content/safeshare_lora', 'zip', '.', 'training/lora_adapter')
shutil.make_archive('/content/safeshare_predictions', 'zip', '.', 'predictions')
from google.colab import files
files.download('/content/safeshare_lora.zip')
files.download('/content/safeshare_predictions.zip')


In [ ]:
# 6 - align with DPO  (~25 min) - starts from the adapter above
os.environ.pop('SAFESHARE2_ADAPTER', None)   # training reads the adapter from --adapter, not the env
!python -m training.train_dpo
!cp -r training/dpo_adapter /content/drive/MyDrive/SafeShare/
print('aligned adapter saved to Drive')


In [ ]:
# 7 - measure after alignment
os.environ['SAFESHARE2_ADAPTER'] = 'training/dpo_adapter'
!python -m evaluation.evaluate --live --out predictions/dpo.jsonl
!python -m evaluation.evaluate --score predictions/dpo.jsonl --json predictions/dpo.metrics.json
!cp predictions/dpo.jsonl predictions/dpo.metrics.json /content/drive/MyDrive/SafeShare/predictions/
print('done')


In [ ]:
# 8 - the table for the report
!python -m evaluation.evaluate --compare predictions/base.jsonl predictions/lora.jsonl predictions/dpo.jsonl

# If you skipped the training, compare what you have:
# !python -m evaluation.evaluate --compare predictions/base.jsonl


In [ ]:
# 9 - final save: always to your computer, to Drive only if it mounted
import os, shutil
from google.colab import files
shutil.rmtree('/content/results', ignore_errors=True)
os.makedirs('/content/results', exist_ok=True)
for name in ('predictions', 'training/lora_adapter', 'training/dpo_adapter'):
    if os.path.exists(name):
        shutil.copytree(name, f'/content/results/{os.path.basename(name)}', dirs_exist_ok=True)
        print('added', name)
shutil.make_archive('/content/safeshare_results', 'zip', '/content/results')
print('zip size:', round(os.path.getsize('/content/safeshare_results.zip') / 1e6, 1), 'MB')
files.download('/content/safeshare_results.zip')
if os.path.isdir('/content/drive/MyDrive'):
    !cp /content/safeshare_results.zip /content/drive/MyDrive/SafeShare/
    print('also copied to Drive')
else:
    print('no Drive - the download above is the copy that matters')


## If something goes wrong

- **"no GPU"** — Runtime → Change runtime type → T4 GPU, then re-run from cell 0.
- **Out of memory during training** — lower the length: `!python -m training.train_lora --max-len 1536`,
  and if it still fails, `--batch-size 1 --grad-accum 16`. Do not trim the policy block out of the
  prompt: that is the part carrying the rules.
- **Disconnected** — Google Drive already holds every predictions file and any adapter that finished.
  Re-run from cell 0; the tooling skips what exists. Training itself has no resume, so start that cell again.
- **The model will not start in cell 4** — the download in cell 3 must finish first; re-run cell 3.
- **Everything finished but the numbers look identical** — that is a legitimate result. Report it, and
  let the prompt-size argument (`docs/MODEL_ADAPTATION.md` §3) carry the case for the training step.
- **Drive mount fails with "credential propagation was unsuccessful"** — a known Colab
  authentication hiccup. Retry with `force_remount=True`; if it still fails, continue without it
  (nothing else needs Drive) and use cell 10b to download the results instead.
- **`operator torchvision::nms does not exist`, then every document fails with `Could not
  import module 'Qwen2ForCausalLM'`** — Colab's torchvision no longer matches the torch this project
  pins. Cell 2 now removes torchvision, which fixes it (nothing here uses images). If you see it anyway:
  `!pip uninstall -y torchvision`, then re-run the cell that failed.


## What to put in the report

The three columns from cell 9, quoted as they print — plus the prompt budget from
`docs/MODEL_ADAPTATION.md` §3: 1,617 characters of instructions + 1,632 of policy for every
798 characters of document. That is the cost the adaptation step is arguing against, and the
baseline column is what makes the argument honest.
